# Saving and Loading Models / 模型的保存与加载
PyTorch provides several methods for saving and loading models. / PyTorch 提供了多种保存和加载模型的方法。

This demo covers several methods using an example model. / 本演示将通过一个示例模型介绍多种方法。

#### Functions for Saving and Loading / 保存与加载相关函数
`torch.save()`: Save PyTorch objects (models, tensors, dictionaries, etc.) using Python's pickle module. / 使用 Python 的 pickle 模块保存 PyTorch 对象（模型、张量、字典等）。

`torch.load()`: Load PyTorch objects into memory. / 将 PyTorch 对象加载到内存中。

`load_state_dict()`: Load saved parameters from objects. / 从保存的对象中加载参数。

In [1]:
# Example fake model / 示例假模型
import torch.nn as nn
import torch.nn.functional as F

class FakeNet(nn.Module):
    def __init__(self):
        super(FakeNet, self).__init__()
        self.fc1 = nn.Linear(10, 50)
        self.batch_norm = nn.BatchNorm1d(50) 
        self.fc2 = nn.Linear(50, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.batch_norm(x)
        x = self.fc2(x)
        return x


In [2]:
# Create our model / 创建模型
model = FakeNet()
print(model)

FakeNet(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (batch_norm): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)


In [3]:
# Create a fake dataset / 创建一个假数据集
import torch
from torch.utils.data import Dataset, DataLoader

class FakeDataset(Dataset):
    def __init__(self, num_samples=1000):
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Generate random input data with 10 features / 生成10维随机输入特征
        x = torch.randn(10)
        # Generate a random target value / 生成随机目标值
        y = torch.randn(1)
        return x, y


# Create dataset and data loader / 创建数据集与数据加载器
dataset = FakeDataset(num_samples=1000)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [4]:
# Create loss function and optimizer / 创建损失函数与优化器
criterion = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [5]:
# Train a fake model / 训练一个假模型
N_EPOCHS = 5

for epoch in range(N_EPOCHS):
    running_loss = 0.0
    for i, (inputs, targets) in enumerate(data_loader):
        # Zero the parameter gradients / 梯度清零
        optimizer.zero_grad()
        
        # Forward pass / 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass and optimize / 反向传播并更新参数
        loss.backward()
        optimizer.step()
        
        # Accumulate loss / 累积损失
        running_loss += loss.item()


# Saving and Loading using `state_dict` / 使用 `state_dict` 保存与加载
`state_dict` is a dictionary that stores all learnable parameters of a model (such as weights and biases), and optimizer hyperparameters. / `state_dict` 是一个字典，用于保存模型中所有可学习参数（如权重和偏置）以及优化器超参数。
This makes saving, loading, and transferring parameters easy across environments. / 这使得在不同环境中保存、加载和迁移参数更加方便。

In [6]:
# Print model state_dict / 打印模型 state_dict
print(model.state_dict())

OrderedDict([('fc1.weight', tensor([[ 2.7979e-01, -1.8727e-01,  3.0561e-01,  2.2709e-01,  9.4731e-02,
          6.0716e-02, -1.4102e-01, -2.9011e-01,  2.9639e-01,  2.4552e-01],
        [ 2.1013e-01,  2.6246e-01, -1.4130e-01,  1.1874e-01, -2.3903e-02,
         -3.0016e-01, -2.0121e-01,  1.1989e-01,  6.9453e-02,  8.4367e-02],
        [-1.8332e-01,  5.8440e-02,  2.5708e-02,  4.0160e-02, -3.0911e-01,
          9.7095e-02,  1.3900e-01, -5.6005e-02,  4.7440e-02, -1.7195e-02],
        [ 1.5695e-01, -1.8205e-01, -2.7385e-01, -1.3724e-02,  2.2638e-02,
         -5.1135e-02,  8.4673e-02,  1.0626e-01,  2.7112e-01, -1.1384e-02],
        [ 2.1908e-01, -3.1170e-01,  8.7608e-02, -1.2941e-01, -3.0754e-02,
         -1.7602e-01, -5.4024e-02, -2.4301e-01, -7.6985e-02,  3.0731e-01],
        [-3.2037e-02,  2.6879e-01, -1.0714e-01, -7.7114e-02, -1.7492e-01,
         -6.5292e-02,  3.9777e-02, -2.8430e-01, -2.2006e-01,  1.8037e-01],
        [ 1.3787e-01, -3.8621e-02, -1.7159e-01,  2.2750e-02, -1.8898e-01,
    

In [7]:
# Print parameters of each layer / 打印每层参数
for k, v in model.state_dict().items():
    print(f"Layer Name / 层名: {k} Parameters / 参数形状: {v.size()}")

Layer Name / 层名: fc1.weight Parameters / 参数形状: torch.Size([50, 10])
Layer Name / 层名: fc1.bias Parameters / 参数形状: torch.Size([50])
Layer Name / 层名: batch_norm.weight Parameters / 参数形状: torch.Size([50])
Layer Name / 层名: batch_norm.bias Parameters / 参数形状: torch.Size([50])
Layer Name / 层名: batch_norm.running_mean Parameters / 参数形状: torch.Size([50])
Layer Name / 层名: batch_norm.running_var Parameters / 参数形状: torch.Size([50])
Layer Name / 层名: batch_norm.num_batches_tracked Parameters / 参数形状: torch.Size([])
Layer Name / 层名: fc2.weight Parameters / 参数形状: torch.Size([1, 50])
Layer Name / 层名: fc2.bias Parameters / 参数形状: torch.Size([1])


In [8]:
# Print optimizer hyperparameters / 打印优化器超参数
print(optimizer.state_dict())

{'state': {}, 'param_groups': [{'lr': 0.01, 'momentum': 0, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'params': [0, 1, 2, 3, 4, 5]}]}


In [9]:
# Save model state_dict (recommended) / 保存模型 state_dict（推荐）
import torch

torch.save(model.state_dict(), "model_state_dict.pt") # .pt or .pth extension / 模型常用扩展名

In [10]:
# Save optimizer state_dict / 保存优化器 state_dict
torch.save(optimizer.state_dict(), "optimizer")

In [11]:
# NOTE: state_dict saves ONLY parameters / 注意：state_dict 只保存参数

### Model Inference / 模型推理
REVIEW: Inference is the process of using a trained model to make predictions. / 复习：推理是使用训练好的模型进行预测的过程。

Let's load a model using its state_dict and prepare it for inference. / 让我们使用 state_dict 加载模型并为推理做好准备。

In [12]:
# Create a new model / 创建新模型
new_model = FakeNet()
print(new_model)

FakeNet(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (batch_norm): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)


In [13]:
# Show current state_dict / 显示当前 state_dict
for k, v in new_model.state_dict().items():
    print(f"Layer Name / 层名: {k} Parameters / 参数: {v}")

Layer Name / 层名: fc1.weight Parameters / 参数: tensor([[ 0.1692,  0.2433, -0.2764,  0.2912, -0.1225, -0.1150, -0.0755, -0.1979,
         -0.1074, -0.3028],
        [-0.1074, -0.1807, -0.1666, -0.0409, -0.2687,  0.1007, -0.0030, -0.0974,
          0.1676, -0.1683],
        [-0.1463,  0.0438,  0.1002,  0.2363,  0.0609,  0.2146,  0.0024, -0.3143,
          0.3098, -0.2224],
        [ 0.0815, -0.0729,  0.0442, -0.1357, -0.2563,  0.0807,  0.2560, -0.1628,
          0.1378, -0.2603],
        [ 0.2931, -0.0884, -0.0082, -0.0644,  0.0536, -0.1887, -0.2028, -0.0686,
          0.2955,  0.1539],
        [ 0.0824, -0.2510, -0.0219, -0.0410,  0.2319,  0.0009, -0.3003,  0.1611,
         -0.3000,  0.2609],
        [-0.1419,  0.1217, -0.0876,  0.2411, -0.0068, -0.0862, -0.0734, -0.2625,
          0.2053,  0.1816],
        [-0.2147,  0.1681,  0.1611, -0.2131,  0.2295,  0.0559,  0.1379, -0.0021,
          0.2663, -0.2339],
        [ 0.0303,  0.2947, -0.2817, -0.0030, -0.2087, -0.3074, -0.2945,  0.0331,
  

In [14]:
# Load parameters into model / 将参数加载到模型
new_model.load_state_dict(torch.load("model_state_dict.pt", weights_only=True)) # ONLY parameters / 仅参数

<All keys matched successfully>

In [15]:
# Print again to show difference / 再次打印以显示差异
for k, v in new_model.state_dict().items():
    print(f"Layer Name / 层名: {k} Parameters / 参数: {v}")

Layer Name / 层名: fc1.weight Parameters / 参数: tensor([[ 2.7979e-01, -1.8727e-01,  3.0561e-01,  2.2709e-01,  9.4731e-02,
          6.0716e-02, -1.4102e-01, -2.9011e-01,  2.9639e-01,  2.4552e-01],
        [ 2.1013e-01,  2.6246e-01, -1.4130e-01,  1.1874e-01, -2.3903e-02,
         -3.0016e-01, -2.0121e-01,  1.1989e-01,  6.9453e-02,  8.4367e-02],
        [-1.8332e-01,  5.8440e-02,  2.5708e-02,  4.0160e-02, -3.0911e-01,
          9.7095e-02,  1.3900e-01, -5.6005e-02,  4.7440e-02, -1.7195e-02],
        [ 1.5695e-01, -1.8205e-01, -2.7385e-01, -1.3724e-02,  2.2638e-02,
         -5.1135e-02,  8.4673e-02,  1.0626e-01,  2.7112e-01, -1.1384e-02],
        [ 2.1908e-01, -3.1170e-01,  8.7608e-02, -1.2941e-01, -3.0754e-02,
         -1.7602e-01, -5.4024e-02, -2.4301e-01, -7.6985e-02,  3.0731e-01],
        [-3.2037e-02,  2.6879e-01, -1.0714e-01, -7.7114e-02, -1.7492e-01,
         -6.5292e-02,  3.9777e-02, -2.8430e-01, -2.2006e-01,  1.8037e-01],
        [ 1.3787e-01, -3.8621e-02, -1.7159e-01,  2.2750e-02, 

In [16]:
# Parameters are updated after loading / 参数在加载后已更新

In [17]:
# Create example input / 创建示例输入
import torch
# Random input: batch size 1, 10 features / 随机输入：batch=1，特征数=10
sample_input = torch.randn(1, 10)
print(sample_input)

tensor([[ 0.4732,  1.5686, -1.5065, -0.5939,  1.4406,  1.6442,  0.2812,  0.0181,
          0.0030, -0.4761]])


In [18]:
# Run an inference example / 执行一次推理示例
new_model.eval()

# Call model with input to get prediction / 输入样本并得到预测
output = new_model(sample_input)
print(output)

tensor([[-0.0024]], grad_fn=<AddmmBackward0>)


# Saving and Loading Entire Model / 保存与加载完整模型
PyTorch also provides the option to save the full model to disk. / PyTorch 也支持将完整模型保存到磁盘。

Full model = full Python pickle version of the model. / 完整模型 = 模型的完整 Python pickle 版本。

This can cause issues because it depends on exact class definitions and file structure at save time. / 这种方式可能带来问题，因为它依赖保存时完全一致的类定义与文件结构。
Loading may fail in different projects or after code changes. / 在不同项目中或代码变动后，加载可能失败。

In [19]:
# Save full model / 保存完整模型
import torch

torch.save(model, "model_full.pt")

In [21]:
# Initialize and use model / 初始化并使用模型
model = FakeNet()
print(model)

FakeNet(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (batch_norm): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)


In [22]:
# Try saving again / 再次尝试保存
torch.save(model, "model_full.pt")

In [24]:
# Compare file sizes / 对比文件大小
# Windows compatible way / Windows 兼容方式
import os
import glob

print("模型文件大小对比:\n")
print(f"{'文件名':<30} {'大小':<15}")
print("-" * 45)

for file in glob.glob('model*'):
    if os.path.isfile(file):
        size = os.path.getsize(file)
        # Convert to human readable format / 转换为可读格式
        if size < 1024:
            size_str = f"{size} B"
        elif size < 1024 * 1024:
            size_str = f"{size / 1024:.2f} KB"
        else:
            size_str = f"{size / (1024 * 1024):.2f} MB"
        print(f"{file:<30} {size_str:<15}")

模型文件大小对比:

文件名                            大小             
---------------------------------------------
model_state_dict.pt            6.55 KB        
model_full.pt                  7.53 KB        


In [26]:
# Load full model workflow / 完整模型加载流程
# Initialize and use model / 初始化并使用模型
new_model = FakeNet()
print(new_model)

FakeNet(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (batch_norm): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)


In [27]:
# Load from full model file / 从完整模型文件加载
new_model = torch.load("model_full.pt", weights_only=False) # More than parameters / 不止参数
print(new_model)

FakeNet(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (batch_norm): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)


In [28]:
# Check inference / 检查推理
new_model.eval()

# Call model with input to get prediction / 输入样本并获取预测
output = new_model(sample_input)
print(output)

tensor([[-0.3019]], grad_fn=<AddmmBackward0>)


# Saving and Loading a Checkpoint / 保存与加载检查点
A checkpoint saves parameters as a snapshot at a point in time. / 检查点是在某个时间点保存参数快照的方式。

This helps resume long training jobs after failures. / 这有助于在训练中断后继续训练。
It also allows selecting among multiple models from one training run. / 也便于从一次训练中保存多个可选模型。

In [29]:
# Save a checkpoint / 保存检查点
import torch

# Dummy epoch and loss / 示例 epoch 与 loss
epoch = 5
loss = 0.05

In [30]:
# Save a checkpoint file / 保存检查点文件
torch.save({'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss}, 
            f'{epoch}_checkpoint.tar') # .tar file / 使用 .tar 文件

In [32]:
# Load checkpoint prerequisites / 加载检查点前置准备
# Initialize model (and optimizer in this case) / 初始化模型（本例还需优化器）

# Initialize and use model / 初始化并使用模型
model = FakeNet()
print(model)

FakeNet(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (batch_norm): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)


In [33]:
# Load model checkpoint / 加载模型检查点
import torch

# Load tar file / 加载 tar 文件
checkpoint = torch.load(f"{epoch}_checkpoint.tar", weights_only='true')

In [34]:
# Show checkpoint info / 显示检查点信息
print(checkpoint)

{'epoch': 5, 'model_state_dict': OrderedDict([('fc1.weight', tensor([[-0.1436, -0.2817,  0.2189, -0.0377,  0.0741, -0.1140,  0.0855, -0.3131,
         -0.1913,  0.1701],
        [ 0.1268, -0.1981, -0.1095, -0.2198, -0.2871,  0.1980,  0.0788, -0.0767,
         -0.1051, -0.1797],
        [ 0.1480, -0.1814,  0.0362,  0.0884, -0.2629, -0.0225, -0.2174, -0.0312,
         -0.2227,  0.2883],
        [-0.1507,  0.2822,  0.2658, -0.2241, -0.2926, -0.2872, -0.0347, -0.1075,
         -0.0345,  0.0996],
        [ 0.1547,  0.1725,  0.2493, -0.2150,  0.1821,  0.2110, -0.1193,  0.1926,
          0.0975,  0.0322],
        [-0.0356, -0.2926, -0.0132, -0.1388,  0.0107, -0.1416,  0.1909,  0.1447,
          0.0461,  0.1153],
        [-0.1833,  0.1371, -0.1563,  0.1018,  0.0223, -0.2818, -0.1802, -0.1741,
         -0.0768, -0.2117],
        [-0.3070, -0.2318, -0.1585, -0.3042, -0.2081, -0.0963,  0.2707, -0.2832,
          0.0587,  0.2582],
        [-0.1168, -0.1923,  0.0969, -0.1948, -0.2453,  0.1926,  0.2

In [35]:
# Load parameters into model / 将参数加载到模型
model.load_state_dict(checkpoint['model_state_dict']) 

<All keys matched successfully>

In [36]:
# Load optimizer state / 加载优化器状态
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [37]:
# Load loss and epoch (you can store more metadata as needed) / 加载 loss 和 epoch（也可按需保存更多信息）
loss = checkpoint['loss']
epoch = checkpoint['epoch']
print(loss, epoch)

0.05 5


In [38]:
# Test inference / 测试推理
model.eval()
output = model(sample_input)
print(output)

tensor([[-0.3019]], grad_fn=<AddmmBackward0>)


# Adding Checkpoints to Training / 在训练中加入检查点
It's good practice to include checkpoints in your training loop. / 在训练循环中加入检查点是一个良好实践。

How to save checkpoints is up to you, e.g., periodically, every epoch, or only when loss improves. / 检查点保存策略可按需选择，例如定期保存、每个 epoch 保存，或仅在损失改进时保存。

In [39]:
# Add checkpoint saving every 2 epochs in training loop / 在训练循环中每2个epoch保存一次检查点
# Train a fake model / 训练一个假模型
N_EPOCHS = 10

for epoch in range(N_EPOCHS):
    running_loss = 0.0
    for i, (inputs, targets) in enumerate(data_loader):
        # Zero the parameter gradients / 梯度清零
        optimizer.zero_grad()
        
        # Forward pass / 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass and optimize / 反向传播并更新参数
        loss.backward()
        optimizer.step()
        
        # Accumulate loss / 累积损失
        running_loss += loss.item()

    ######### Save checkpoint every 2 epochs / 每2个epoch保存检查点
    if epoch % 2 == 0:
        torch.save({'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss}, 
                f'training_checkpoint_{epoch}.tar')

# Save final checkpoint after last epoch / 最后一个epoch后保存最终检查点
torch.save({
    'epoch': N_EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss
}, 'training_checkpoint_final.tar')

In [40]:
# List all checkpoints / 列出所有检查点文件
# Windows compatible way / Windows 兼容方式
import os
import glob

print("检查点文件列表:\n")
print(f"{'文件名':<40} {'大小':<15} {'修改时间'}")
print("-" * 75)

for file in sorted(glob.glob('training_checkpoint*')):
    if os.path.isfile(file):
        size = os.path.getsize(file)
        # Convert to human readable format / 转换为可读格式
        if size < 1024:
            size_str = f"{size} B"
        elif size < 1024 * 1024:
            size_str = f"{size / 1024:.2f} KB"
        else:
            size_str = f"{size / (1024 * 1024):.2f} MB"
        
        # Get modification time / 获取修改时间
        from datetime import datetime
        mtime = datetime.fromtimestamp(os.path.getmtime(file))
        time_str = mtime.strftime('%Y-%m-%d %H:%M:%S')
        
        print(f"{file:<40} {size_str:<15} {time_str}")

检查点文件列表:

文件名                                      大小              修改时间
---------------------------------------------------------------------------
training_checkpoint_0.tar                7.12 KB         2026-03-03 16:14:48
training_checkpoint_2.tar                7.12 KB         2026-03-03 16:14:48
training_checkpoint_4.tar                7.12 KB         2026-03-03 16:14:48
training_checkpoint_6.tar                7.12 KB         2026-03-03 16:14:48
training_checkpoint_8.tar                7.12 KB         2026-03-03 16:14:48
training_checkpoint_final.tar            7.24 KB         2026-03-03 16:14:48


NOTE: We can now load any of these checkpoints to continue training from that point or run inference. / 注意：我们现在可以加载任意检查点，从该时间点继续训练或直接执行推理。

# Warmstarting / 热启动
Warmstarting means initializing a new model with parameters from a previously trained model. / 热启动是指用已训练模型的参数来初始化新模型。

This is very helpful in transfer learning (covered in detail later). / 这在迁移学习中非常有用（后续会详细介绍）。

With warmstarting, we can also initialize only selected layers from a trained model. / 使用热启动时，也可以只初始化已训练模型中的部分层。

In [41]:
# Example fake model / 示例假模型
import torch.nn as nn
import torch.nn.functional as F

class FakeNet(nn.Module):
    def __init__(self):
        super(FakeNet, self).__init__()
        self.fc1 = nn.Linear(10, 50)
        self.batch_norm = nn.BatchNorm1d(50) 
        self.fc2 = nn.Linear(50, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.batch_norm(x)
        x = self.fc2(x)
        return x

In [42]:
# Create new model / 创建新模型
new_model = FakeNet()
new_model

FakeNet(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (batch_norm): BatchNorm1d(50, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)

In [43]:
# Show parameters / 显示参数
new_model.state_dict()

OrderedDict([('fc1.weight',
              tensor([[-0.0300,  0.0846,  0.3116, -0.1158, -0.2587, -0.0268, -0.3054,  0.2916,
                       -0.2160, -0.1778],
                      [-0.0806,  0.0386,  0.2351,  0.0313,  0.1303, -0.1147,  0.2319,  0.0539,
                       -0.1036, -0.0648],
                      [-0.1172, -0.2190, -0.1772, -0.2550, -0.2963,  0.1023,  0.1058, -0.0591,
                        0.1204, -0.0013],
                      [ 0.2681, -0.0057, -0.0718,  0.1232, -0.1638,  0.2055, -0.1369, -0.2465,
                       -0.1173, -0.0545],
                      [-0.1381,  0.2729, -0.2650,  0.2421,  0.0052,  0.0248,  0.0933, -0.0862,
                       -0.0270, -0.2031],
                      [-0.2844, -0.0637, -0.3158, -0.0646,  0.1205, -0.1122, -0.2177,  0.1780,
                        0.2416, -0.2184],
                      [-0.2333, -0.0834,  0.1759, -0.1323,  0.0021,  0.2905, -0.1458, -0.0700,
                        0.0939, -0.0151],
             

In [44]:
# Load first trained model parameters into new model / 将最初训练好的参数加载到新模型
new_model.load_state_dict(torch.load('model_state_dict.pt'), strict=False)

C:\Users\LIHAN\AppData\Local\Temp\ipykernel_4116\26893491.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  new_model.load_state_dict(torch.load('model_state_dict.pt'), st

<All keys matched successfully>

In [45]:
# Show updated parameters / 显示更新后的参数
print(new_model.state_dict())

OrderedDict([('fc1.weight', tensor([[ 2.7979e-01, -1.8727e-01,  3.0561e-01,  2.2709e-01,  9.4731e-02,
          6.0716e-02, -1.4102e-01, -2.9011e-01,  2.9639e-01,  2.4552e-01],
        [ 2.1013e-01,  2.6246e-01, -1.4130e-01,  1.1874e-01, -2.3903e-02,
         -3.0016e-01, -2.0121e-01,  1.1989e-01,  6.9453e-02,  8.4367e-02],
        [-1.8332e-01,  5.8440e-02,  2.5708e-02,  4.0160e-02, -3.0911e-01,
          9.7095e-02,  1.3900e-01, -5.6005e-02,  4.7440e-02, -1.7195e-02],
        [ 1.5695e-01, -1.8205e-01, -2.7385e-01, -1.3724e-02,  2.2638e-02,
         -5.1135e-02,  8.4673e-02,  1.0626e-01,  2.7112e-01, -1.1384e-02],
        [ 2.1908e-01, -3.1170e-01,  8.7608e-02, -1.2941e-01, -3.0754e-02,
         -1.7602e-01, -5.4024e-02, -2.4301e-01, -7.6985e-02,  3.0731e-01],
        [-3.2037e-02,  2.6879e-01, -1.0714e-01, -7.7114e-02, -1.7492e-01,
         -6.5292e-02,  3.9777e-02, -2.8430e-01, -2.2006e-01,  1.8037e-01],
        [ 1.3787e-01, -3.8621e-02, -1.7159e-01,  2.2750e-02, -1.8898e-01,
    

We can now train this model using the loaded parameters. / 现在我们可以基于已加载参数继续训练这个模型。

# Saving and Loading Across Devices / 跨设备保存与加载
PyTorch supports different devices such as CPU and GPU. / PyTorch 支持 CPU、GPU 等不同设备。

A common practice is training on GPU for speed and doing inference on CPU for cost efficiency. / 常见实践是用 GPU 加速训练，用 CPU 进行低成本推理。

In [46]:
# Load parameters on CPU from GPU-saved file / 将GPU保存的参数映射到CPU加载
import torch

model = torch.load('model_state_dict.pt', map_location='cpu', weights_only=True) # Using map_location / 使用 map_location

In [47]:
# CPU to GPU and GPU to GPU mapping / CPU到GPU或GPU到GPU映射
model = torch.load('model_state_dict.pt', map_location='cuda:0', weights_only=True) # Map to GPU device / 映射到GPU设备

In addition to the above, we must also move the model to GPU. / 除了上面的步骤，我们还需要把模型移动到 GPU：

```py
model.to('cuda')
```

As well as the inputs during inference. / 推理时输入也要放到同一设备上：
```py
model.eval()
outputs = model(sample_input.to('cuda'))
```

In [48]:
# Remember is_available() to detect device / 使用 is_available() 检测设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda
